# CS 450 · How a model learns from labeled examples (Iris)

**About 20 minutes. You do not need to know Python.**

Open this file in [Google Colab](https://colab.research.google.com/)
(`File → Upload notebook`), then use **Runtime → Run all**.

You will:

1. See what each **label** looks like (a photo of that iris species) next to
   **real rows** from the dataset (four measurements).
2. **Train** (make) a simple model from some of those rows.
3. **Infer** (use) it on flowers it did not train on, and read one **accuracy**.
4. Change **one** setting — tree depth — and see the score on new flowers improve.

This is **supervised learning**: each example already has the right species name.
The model learns from **numbers** (petal and sepal size), not from the photos.
The photos are so the label is not an abstract word.

This is not a course in building models. Other CS courses teach that. Today you
see enough of training to know what you are *using* later with tools like ChatGPT.

Handwritten digits / MNIST are **not** this notebook. That set waits for neural
nets later.


## Step 0 — Load the tools

Colab already includes these libraries. Run the cell. You should see `ready`.


In [ ]:
# If an import ever fails in Colab, uncomment the next line and run this cell again.
# %pip install -q scikit-learn matplotlib pandas pillow

from io import BytesIO
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

print("ready")


## Step 1 — Load the Iris data

**Iris** is a classic beginner set: 150 flowers, three species, 50 of each.

Each **example** is one flower: four measurements (cm) plus a **label** (the
species). That label is what we want the model to predict.

| Feature (input) | Meaning |
|---|---|
| sepal length / width | Outer flower parts that protect the bud |
| petal length / width | Inner, often colorful flower parts |
| **label** | `setosa`, `versicolor`, or `virginica` |


In [ ]:
iris = load_iris()

flowers = pd.DataFrame(iris.data, columns=iris.feature_names)
flowers["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"{len(flowers)} labeled flowers")
print("Species:", list(iris.target_names))
print("Counts:\n", flowers["species"].value_counts().to_string())
print()
print("First six rows — these are the actual dataset examples:")
flowers.head(6)


## Step 2 — See the picture that belongs to each label

The dataset itself is a table, not a photo album. To make the **label**
concrete, we show one photograph of each species — the same three names that
appear in the `species` column.

Then we pick **one real row per species** from the table and print the rows
below the photos. The photos illustrate each species; **they are not photos
of the individual flowers measured in the dataset**. The numbers are real
dataset rows, but there is no photo-to-measurement match.

If a photo cannot load, its species name appears instead. The measurements,
training, and scores still work.


In [ ]:
# Wikimedia Commons thumbnails (CC-licensed photographs). Used only to show
# what each species name refers to. The model below does not train on pixels.
SPECIES_PHOTOS = {
    "setosa": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/"
        "5/56/Kosaciec_szczecinkowaty_Iris_setosa.jpg/330px-Kosaciec_szczecinkowaty_Iris_setosa.jpg"
    ),
    "versicolor": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/"
        "4/41/Iris_versicolor_3.jpg/330px-Iris_versicolor_3.jpg"
    ),
    "virginica": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/"
        "9/9f/Iris_virginica.jpg/330px-Iris_virginica.jpg"
    ),
}


PHOTO_CACHE = {}


def load_photo(url):
    if url not in PHOTO_CACHE:
        try:
            req = Request(
                url,
                headers={"User-Agent": "CS450-IrisNotebook/1.0 (UNH teaching notebook)"},
            )
            with urlopen(req, timeout=10) as response:
                PHOTO_CACHE[url] = Image.open(BytesIO(response.read())).convert("RGB")
        except (OSError, ValueError) as error:
            print(f"Photo unavailable ({type(error).__name__}); continuing with labels.")
            PHOTO_CACHE[url] = None
    return PHOTO_CACHE[url]


def show_species_photo(ax, species):
    photo = load_photo(SPECIES_PHOTOS[species])
    if photo is None:
        ax.text(0.5, 0.5, f"{species}\n(photo unavailable)",
                ha="center", va="center", transform=ax.transAxes)
    else:
        ax.imshow(photo)


fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
fig.suptitle("Each label is a species — here is what that flower looks like", y=1.02)

example_rows = []
for ax, species in zip(axes, iris.target_names):
    show_species_photo(ax, species)
    ax.set_title(f"label: {species}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    # one real dataset row for this species
    row = flowers[flowers["species"] == species].iloc[0]
    example_rows.append(row)

plt.tight_layout()
plt.show()

print("One actual dataset example for each label (measurements in cm):")
pd.DataFrame(example_rows).reset_index(drop=True)


## Step 3 — See every example as a picture of measurements

Each point is one flower from the table. Color = **label**. You can see that
`setosa` separates cleanly; the other two overlap a little — which is why a
too-simple rule will mix them up.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for species, color in zip(iris.target_names, ["#1b9e77", "#d95f02", "#7570b3"]):
    part = flowers[flowers["species"] == species]
    ax.scatter(
        part["petal length (cm)"],
        part["petal width (cm)"],
        label=species,
        s=40,
        alpha=0.85,
        c=color,
    )
ax.set_xlabel("petal length (cm)")
ax.set_ylabel("petal width (cm)")
ax.set_title("All 150 dataset examples (color = label)")
ax.legend(title="label")
plt.show()


## Step 4 — What a **model** is here

A **model** is software fitted from data that can predict a label for a new
example.

- **Training** = *make* the model: show it labeled rows so it can form a rule.
- **Inference** = *use* the model: give it new measurements and read its species guess.

The model we fit is a **decision tree**: a flowchart of yes/no questions about
the four measurements (for example, “is petal length less than 2.5 cm?”).
You do not write that flowchart. Training does.

### The one knob you will change

**`TREE_DEPTH`** = how many layers of questions the tree is allowed to ask.

| Setting | What it means | What you should expect |
|---|---|---|
| `1` (start here) | One question — too simple | Mixes versicolor and virginica |
| `3` (try next) | A few more questions — enough for this small set | Clearly better on new flowers |

Goldilocks from lecture: not enough fit vs a better fit. Do not hunt for a
perfect number. Try **1**, then **3**.


## Step 5 — Hold some flowers out

If we tested only on flowers the tree had already seen, a high score could mean
“it memorized this batch.” We **hold out** about one quarter of the rows and
never train on them. The score later is only on those **new** flowers.

You do not need two scores. One score on new cases is the check.


In [ ]:
# random_state=7 means everyone in class gets the same split.
meas_train, meas_new, labels_train, labels_new = train_test_split(
    iris.data,
    iris.target,
    test_size=0.25,
    random_state=7,
    stratify=iris.target,
)

print(f"Used to train (make) the model: {len(labels_train)} flowers")
print(f"Held out to check (new cases):  {len(labels_new)} flowers")
print("New-case labels:", np.bincount(labels_new), "in order", list(iris.target_names))


## Step 6 — Pick the knob, then train

**This is the only line you are asked to edit.** Leave it at `1` for the first
run. After you read the accuracy in Step 7, change it to `3`, then
**Runtime → Run all** again (or re-run this cell and every cell below it).


In [ ]:
# ▼▼▼  THE KNOB  ▼▼▼
TREE_DEPTH = 1    # first run: 1     second run: 3
# ▲▲▲  change only this number  ▲▲▲

print(f"TREE_DEPTH = {TREE_DEPTH}")
print("Training…")

model = DecisionTreeClassifier(max_depth=TREE_DEPTH, random_state=7)
model.fit(meas_train, labels_train)  # training: make the model from labeled rows

print("Training finished. The model can now be used on new flowers.")


## Step 7 — Use the model on new flowers (inference) and read one score

**Accuracy** here means: on the held-out flowers only,

\(\text{accuracy} = \dfrac{\text{how many species guesses matched the label}}{\text{how many new flowers}}\)

It is not “the model knows every iris on earth.” It is a count on **this**
held-out set.


In [ ]:
predicted = model.predict(meas_new)  # inference: use the model

n_new = len(labels_new)
n_correct = int(np.sum(predicted == labels_new))
accuracy = n_correct / n_new

print(f"TREE_DEPTH = {TREE_DEPTH}")
print(f"New flowers checked: {n_new}")
print(f"Correct predictions: {n_correct}")
print(f"Accuracy:            {n_correct}/{n_new}  =  {accuracy:.1%}")
print()
print("Expected ballpark with this notebook's split:")
print("  TREE_DEPTH = 1  →  25/38  (about 66%, too simple)")
print("  TREE_DEPTH = 3  →  37/38  (about 97%, clearly better)")


## Step 8 — Look at new cases: measurements, true label, guess

Each row is a held-out flower (not used in training). The thumbnail is the
illustration of the **true** species, not a photograph of that measured flower.
Green guess = match. Red = miss.


In [ ]:
show_n = min(9, n_new)
fig, axes = plt.subplots(3, 3, figsize=(11, 10))
fig.suptitle(
    f"New flowers · TREE_DEPTH = {TREE_DEPTH} · green = correct, red = miss",
    y=1.01,
    fontsize=12,
)

for i, ax in enumerate(axes.flat):
    true_name = iris.target_names[labels_new[i]]
    pred_name = iris.target_names[predicted[i]]
    ok = labels_new[i] == predicted[i]
    show_species_photo(ax, true_name)
    row = meas_new[i]
    ax.set_title(
        f"true {true_name}\npred {pred_name}\n"
        f"petal {row[2]:.1f}×{row[3]:.1f} cm",
        fontsize=9,
        color="green" if ok else "crimson",
    )
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

checked = pd.DataFrame(meas_new[:show_n], columns=iris.feature_names)
checked["true label"] = [iris.target_names[i] for i in labels_new[:show_n]]
checked["prediction"] = [iris.target_names[i] for i in predicted[:show_n]]
checked["match?"] = np.where(labels_new[:show_n] == predicted[:show_n], "yes", "NO")
checked


## Step 9 — Change the knob (the improvement)

1. Scroll back to **Step 6**.
2. Set `TREE_DEPTH = 3`.
3. Run **from that cell downward** (or **Runtime → Run all**).
4. Compare the Step 7 accuracy and the Step 8 grid.

You should see **more green** and a **higher** correct-count on the same
held-out flowers. Depth `1` was too simple (it mostly separates setosa from
“everything else”). Depth `3` has enough questions to split versicolor from
virginica on this set.

You are not being asked to search for a best depth, to compare a training
score, or to read a loss number. One knob, one check on new cases.


## Step 10 — Record what this does *not* establish

Fill in after your **depth 3** run:

- Model: decision tree. `TREE_DEPTH` I used: ____
- Held-out flowers: ____  Correct: ____  Accuracy: ____
- What improved when I raised the depth: ____
- This score does **not** establish: that the model can name every iris in the
  world, or that a chatbot “knows” plants. It is a count on **these** held-out
  flowers, using four measurements.

We compared two depths on this same held-out group. It remains separate from
training, but it has now helped us compare settings. To check a model chosen
this way independently, we would need another untouched group of flowers.

If Colab will not run, use this paper path for the same split:

| `TREE_DEPTH` | Correct / 38 new flowers | Accuracy |
|---:|---|---:|
| 1 | 25 / 38 | 65.8% |
| 3 | 37 / 38 | 97.4% |


## Step 11 — If you want to *make* models later

You just **used** a training procedure someone else wrote. Designing models,
choosing data, and testing them carefully is what later CS / data courses are
for.

What this course needed: you can see labeled examples, you know a **model** is
fitted then used, and you know a score is about **new cases**, not omniscience.
Neural nets (and MNIST-style picture models) use the same logic at larger
scale — that is next, not this notebook.


## Photo credits

Species illustrations from Wikimedia Commons, displayed at reduced size;
no cropping or other edits in this notebook. They are not dataset specimens.

- **Setosa:** [Radomil, *Kosaciec szczecinkowaty Iris setosa*](https://commons.wikimedia.org/wiki/File:Kosaciec_szczecinkowaty_Iris_setosa.jpg), [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).
- **Versicolor:** [Danielle Langlois, *Iris versicolor 3*](https://commons.wikimedia.org/wiki/File:Iris_versicolor_3.jpg), [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).
- **Virginica:** [Frank Mayfield, *Iris virginica*](https://commons.wikimedia.org/wiki/File:Iris_virginica.jpg), [CC BY-SA 2.0](https://creativecommons.org/licenses/by-sa/2.0/).
